# Разработка алгоритма языковой диаризации для решения проблемы мультиязычности в автоматическом распознавании речи

## 1. Обучение модели языковой идентификации

В качестве базовой архитектуры для обучения модели идентификации языка разговорной речи используется энкодер модели whisper-tiny.

In [3]:
from __future__ import annotations

import io
from enum import Enum
from pathlib import Path
from typing import Iterable, Union, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.nn.functional import relu
from whisper import _download, _MODELS
from whisper.model import (
    Conv1d,
    ResidualAttentionBlock,
    LayerNorm,
    sinusoids,
    ModelDimensions,
)

Код инференса модели модифицирован таким образом, чтобы обойти ограничение в 30 секунд.

In [4]:
class WhisperEncoder(nn.Module):
    def __init__(
        self, n_mels: int, n_ctx: int, n_state: int, n_head: int, n_layer: int
    ):
        super().__init__()
        self.conv1 = Conv1d(n_mels, n_state, kernel_size=3, padding=1)
        self.conv2 = Conv1d(n_state, n_state, kernel_size=3, stride=2, padding=1)
        self.register_buffer("positional_embedding", sinusoids(n_ctx, n_state))

        self.blocks: Iterable[ResidualAttentionBlock] = nn.ModuleList(
            [ResidualAttentionBlock(n_state, n_head) for _ in range(n_layer)]
        )
        self.ln_post = LayerNorm(n_state)

    def forward(self, x: Tensor):
        """
        x : torch.Tensor, shape = (batch_size, n_mels, n_ctx)
            the mel spectrogram of the audio
        """
        x = x.squeeze(1)
        x = F.gelu(self.conv1(x))
        x = F.gelu(self.conv2(x))
        x = x.permute(0, 2, 1)

        x = (x + self.positional_embedding[:x.shape[1], ...]).to(x.dtype)

        for block in self.blocks:
            x = block(x)

        x = self.ln_post(x)

        x = x.permute((0, 2, 1))
        return x

    @staticmethod
    def build_model(
        model_name: str,
        root: Union[str, Path] = Path("./models_zoo/whisper"),
        load_weight=True,
        n_layer: int = 0,
    ):
        if isinstance(root, str):
            root = Path(root)
        if not (root / f"{model_name}.pt").exists():
            checkpoint_file = _download(_MODELS[model_name], root, True)
        else:
            checkpoint_file = open(root / f"{model_name}.pt", "rb").read()

        with io.BytesIO(checkpoint_file) as fp:
            checkpoint = torch.load(fp)
        del checkpoint_file

        dims = ModelDimensions(**checkpoint["dims"])
        model = WhisperEncoder(
            n_mels=dims.n_mels,
            n_ctx=dims.n_audio_ctx,
            n_state=dims.n_audio_state,
            n_head=dims.n_audio_head,
            n_layer=dims.n_audio_layer + n_layer,
        )
        if load_weight:
            filter_encoder = {
                key.replace("encoder.", "", 1): val
                for key, val in checkpoint["model_state_dict"].items()
                if "encoder" in key
            }
            model.load_state_dict(filter_encoder, strict=False)
        model.train()
        model.requires_grad_(True)

        return model

В качестве входных признаков для обучения модели используются 80-размерные фильтр-банки.

In [5]:
import torch
import torch.nn.functional as F
import torchaudio
from torch_audiomentations.core.transforms_interface import BaseWaveformTransform
from whisper.audio import (
    N_SAMPLES,
    # N_MELS,
    N_FFT,
    HOP_LENGTH,
    mel_filters,
)


def pad_or_trim(
    array, length: int = N_SAMPLES, *, axis: int = -1
):
    """
    Pad or trim the audio array to N_SAMPLES, as expected by the encoder.
    """
    if array.shape[axis] > length:
        array = array.index_select(
            dim=axis,
            index=torch.arange(length, device=array.device),
        )

    if array.shape[axis] < length:
        pad_widths = [(0, 0)] * array.ndim
        pad_widths[axis] = (0, length - array.shape[axis])
        array = F.pad(
            array,
            [
                pad
                for sizes in pad_widths[::-1]
                for pad in sizes
            ],
        )
    return array


def trim(array, length: int = N_SAMPLES, *, axis: int = -1):
    """
    Pad or trim the audio array to N_SAMPLES, as expected by the encoder.
    """
    if array.shape[axis] > length:
        array = array.index_select(
            dim=axis,
            index=torch.arange(length, device=array.device),
        )
    return array


def log_mel_spectrogram(
    audio: torch.Tensor,
    # n_mels: int = N_MELS,
    n_mels: int = 80
):
    window = torch.hann_window(N_FFT).to(audio.device)
    stft = torch.stft(
        audio,
        N_FFT,
        HOP_LENGTH,
        window=window,
        return_complex=True,
    )
    magnitudes = stft[..., :-1].abs() ** 2

    filters = mel_filters(audio.device, n_mels)
    mel_spec = filters @ magnitudes

    log_spec = torch.clamp(mel_spec, min=1e-10).log10()
    log_spec = torch.maximum(log_spec, log_spec.max() - 8.0)
    log_spec = (log_spec + 4.0) / 4.0
    return log_spec


def wav_normalization(wav, sample_rate=None, squeeze=False):
    if wav.max() > 1 or wav.min() < -1:
        wav = wav / (2**15)
    if squeeze:
        wav = wav[0]
    return wav


class ExtractWhisperFbanks80(BaseWaveformTransform):
    def __init__(self, pad_data: bool, add_batch_dim: bool = True, time_last_dim: bool = True) -> None:
        super().__init__(p=1.0)
        self.add_batch_dim = add_batch_dim
        self.time_last_dim = time_last_dim

        if pad_data:
            self.pipeline = [
                wav_normalization,
                pad_or_trim,
                log_mel_spectrogram,
            ]
        else:
            self.pipeline = [
                wav_normalization,
                trim,
                log_mel_spectrogram,
            ]

    def forward(
        self, samples: torch.Tensor, sample_rate: int = 16_000, **kwargs
    ) -> torch.Tensor:
        if sample_rate != 16_000:
            samples = torchaudio.transforms.Resample(
                orig_freq=sample_rate, new_freq=16_000
            )(samples)
        if len(samples.shape) >= 2:
            samples = samples.squeeze()
        for step in self.pipeline:
            samples = step(samples)

        if self.add_batch_dim:
            samples = samples.unsqueeze(0)

        if not self.time_last_dim:
            samples = samples.swapaxes(-1, -2)

        return samples


Для балансировки классов используется сэмплировщик, представленный в статье Scaling Speech Technology to 1,000+ Languages.

In [6]:
from typing import Union

import pandas as pd
from torch.utils.data import WeightedRandomSampler

from src.data.datasets.base_dataset import WavDataset, ConcatWavDataset


class MMSSampler(WeightedRandomSampler):
    """Samples elements from several datasets like in https://arxiv.org/pdf/2305.13516.pdf (described in part 4.1).

    Two data sampling steps are performed:
     first, for each dataset, data is sampled for the different languages L from a distribution pl ∼  (nl/N)^βL,
     where l = 1, . . . , L, nl is the amount of unlabeled data for each language in the dataset, N is the total
     amount of training in the dataset, and βL is the upsampling factor which controls the trade-off between high-
     and low-resource languages during pretraining. Second, different datasets are balanced by treating each dataset
    as a language in the above sampling scheme with a sampling parameter βD.

    Args:
        dataset (LIDDataset or LIDConcatDataset): dataset for sampling, needs metadata field.
        language_pow (float): βL from paper, authors use 0.5 value
        dataset_pow (float): βD from paper, authors use 0.5 value
        batch_size (int): number of samples in one mini-batch
        epoch_length (int): number of batches in one epoch, if None all samples from dataset will be sampled
    """

    def __init__(self, dataset: Union[WavDataset, ConcatWavDataset],
                 language_pow: float, dataset_pow: float):
        if isinstance(dataset, WavDataset):
            datasets = [dataset]
        elif isinstance(dataset, ConcatWavDataset):
            datasets = dataset.datasets
        else:
            raise TypeError("Wrong type of dataset: should be LIDDataset or LIDConcatDataset.")

        num_samples = len(dataset)
        dataset_weights = [(len(d) / num_samples) ** dataset_pow / len(d) for d in datasets]
        weights = []

        for w, d in zip(dataset_weights, datasets):
            _classes = []
            for index in range(len(d)):
                key = index if d.index2key_dict is None else d.index2key_dict[index]
                _classes.append(d.label_dict[key])
            d_m = pd.DataFrame({'label': _classes})
            sample_weights = d_m['label'].value_counts(normalize=True)
            sample_weights **= language_pow
            sample_weights /= d_m['label'].value_counts()
            sample_weights *= w
            weights.extend(d_m['label'].map(sample_weights).tolist())

        super().__init__(weights, num_samples)


В качестве функции потерь используется AAM-SoftMax.

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class AAMSoftmax(nn.Module):
    def __init__(self, embeding_size, num_classes, margin=0.3, scale=15, **kwargs):
        super(AAMSoftmax, self).__init__()

        self.test_normalize = True

        self.m = margin
        self.s = scale
        self.in_feats = embeding_size
        self.weight = torch.nn.Parameter(torch.FloatTensor(num_classes, embeding_size), requires_grad=True)
        self.ce = nn.CrossEntropyLoss()
        nn.init.xavier_normal_(self.weight, gain=1)

        self.cos_m = math.cos(self.m)
        self.sin_m = math.sin(self.m)

        # make the function cos(theta+m) monotonic decreasing while theta in [0°,180°]
        self.th = math.cos(math.pi - self.m)
        self.mm = math.sin(math.pi - self.m) * self.m

        print('Initialised AAMSoftmax margin %.3f scale %.3f' % (self.m, self.s))

    def forward(self, x, label=None):
        # x : [batch, in_feats]
        # len(label) : batch

        # cos(theta)
        cosine = F.linear(F.normalize(x), F.normalize(self.weight))
        # cos(theta + m)
        sine = torch.sqrt((1.0 - torch.mul(cosine, cosine)).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where((cosine - self.th) > 0, phi, cosine - self.mm)

        # one_hot = torch.zeros(cosine.size(), device='cuda' if torch.cuda.is_available() else 'cpu')
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output = output * self.s

        return output

Для обучения используется набор датасетов, который можно видеть на части конфигурационного файла ниже.

Обучение модели происходит с помощью пайплайна обучения.

In [13]:
!python3 src/train.py -cn lid-whisper

/home/testuser/env_p310/lib/python3.10/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'lid-whisper': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
Seed set to 42
[2025-05-22 01:42:17,366][__main__][INFO] - [rank: 0] Instantiating datamodule <src.data.modules.clearml.ClearMLDataModule>
preprocessor_config.json: 100%|███████████████| 185k/185k [00:00<00:00, 268kB/s]
[2025-05-22 01:42:21,185][__main__][INFO] - [rank: 0] Instantiating experiment <src.experiment.classifier_experiment.ClassificationTaskModule>
Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 13] Permission denied: '/mnt/cs/voice/marchevskiy/hf_cache/models--openai--whisper-small/.no_exist'
[2025-05-22 01:42:22,561][huggingface_hub.file_download][ERROR] - Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 13] Permissio

Ниже представлены валидационные кривые по метрике точность для некоторых тестовых баз данных.

![title](img/1.png)

Ниже представлены некоторые результаты по точности работы модели.

## 2. Языковая диаризация

Алгоритм диаризации состоит из двух главных этапов - извлечение признаков скользящим окном и кластеризация полученных результатов. Ниже представлены классы, реализующие логику работы данных шагов.

In [16]:
import sys
from typing import Tuple, List, Dict
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np
from pyannote.core import Annotation, Segment

from language_diarization.core.modules.base import AcousticModule

from language_diarization.utils.vad import read_vad
from language_diarization.utils.emb import write_embeddings
from language_diarization.utils.enum import EmbeddingType
from language_diarization.utils.meta import MetasDirReader, create_metas_dir
from language_diarization.utils.basic import *
from language_diarization.utils.tensor import split_to_chunks
from language_diarization.utils.constants import FRAME_LENGTH
from language_diarization.utils.annotation import *


class EmbeddingsExtractionPipeline:
    def __init__(
        self,
        module: AcousticModule,
        embeddings_type: str = "NORM_SIMPLE",
        vad_collar: float = 0.0,
        speech_duration_thresholds: Tuple[float] = [0.0, float("inf")],
        chunk_size: float = 1.5,
        chunk_step: float = 0.75,
        intersect_markups: bool = False,
        device: str = "cpu",
        random_seed: int = None,
    ):
        self.module = module
        self.embeddings_type = embeddings_type
        self.vad_collar = vad_collar
        self.speech_dur_thr = speech_duration_thresholds
        self.chunk_size = chunk_size
        self.chunk_step = chunk_step
        self.intersect_markups = intersect_markups
        self.device = device

        seed_everything(random_seed)

    def _read_ref_rttm(self, data_item: Dict) -> Annotation:
        """Reads reference rttm considering rttm key name"""

        rttm_keys = ["rttm", "rttm_lang"]
        ref_annot = None
        for key in rttm_keys:
            if data_item.get(key, None):
                ref_annot = read_rttm(data_item[key])
                ref_annot = filter_segments_by_duration(ref_annot, FRAME_LENGTH)
                break

        if ref_annot is None:
            print(
                f"""
                Reference RTTM didn't provided for {data_item['wav']}.
                Verify than one of next meta files used in metas-reader: {rttm_keys}.
                """
            )

        return ref_annot

    def _postprocess_extractor_output(
        self, extractor_output: Tuple[torch.Tensor]
    ) -> np.ndarray:
        """Chooses and normalizes specified embeddings"""

        def normalize(data: torch.Tensor):
            try:
                result = F.normalize(data, dim=1)
            except AttributeError:
                result = None

            return result

        emb_type = EmbeddingType[self.embeddings_type]
        if emb_type == EmbeddingType.CLASS:
            result = extractor_output[1]
        elif emb_type == EmbeddingType.NORM_CLASS:
            result = normalize(extractor_output[1])
        elif emb_type == EmbeddingType.SIMPLE:
            result = extractor_output[0]
        elif emb_type == EmbeddingType.NORM_SIMPLE:
            result = normalize(extractor_output[0])

        if result is None:
            module_name = self.module.__class__.__name__
            msg = f"Error: Selected module ({module_name}) does not support {emb_type} embeddings"
            print(msg)
            sys.exit(1)

        return result.detach().cpu().numpy()

    def _extract_embeddings(
        self, wav: torch.Tensor, vad: Annotation, sample_rate: int
    ) -> Tuple[np.ndarray, List[Segment]]:
        """Extracts chunked embeddings from input audio"""

        segments = []
        self.module.eval()
        self.module.to(self.device)
        with torch.no_grad():
            embeddings = []
            for seg in vad.itersegments():
                # Split to chunks
                start, end = int(seg.start * sample_rate), int(seg.end * sample_rate)
                part = wav[..., start:end]
                chunks, _ = split_to_chunks(
                    input=part,
                    chunk_size=int(self.chunk_size * sample_rate),
                    chunk_step=int(self.chunk_step * sample_rate),
                )

                # Extract features
                feature_chunks = [
                    self.module.prepare_data(ch, sample_rate) for ch in chunks
                ]
                feature_chunks = torch.stack(feature_chunks, dim=0)

                # Extract embeddings for each chunk
                output = self.module.extract_embeddings(feature_chunks.to(self.device))
                embs = self._postprocess_extractor_output(output)
                embeddings.append(embs)

                # Map each embedding on time axis
                for i in range(len(embs)):
                    start = seg.start + i * self.chunk_step
                    end = start + self.chunk_size
                    segments.append(Segment(start, end))

            embeddings = np.vstack(embeddings)

        # Empty GPU
        if self.device != "cpu":
            self.module = self.module.cpu()
            torch.cuda.empty_cache()

        return embeddings, segments

    def extraction_step(
        self,
        wav_path: str,
        vad_path: str,
        ref_annot: Annotation = None,
    ) -> Annotation:
        wav, sr = self.module.read_data(wav_path)
        vad_annot = read_vad(
            vad_path=vad_path,
            expected_size=wav.shape[-1],
            sample_rate=sr,
        )
        vad_annot = filter_segments_by_duration(vad_annot, self.speech_dur_thr)
        vad_annot = add_collar(vad_annot, self.vad_collar)

        if ref_annot and self.intersect_markups:
            vad_annot, ref_annot = intersect(vad_annot, ref_annot)

        return self._extract_embeddings(wav, vad_annot, sr)

    def run(self, wav_path: str, vad_path: str) -> Annotation:
        """Runs embeddings extraction pipeline on specified wav+vad pair."""

        return self.extraction_step(wav_path, vad_path)

    def run_over_metas(
        self,
        metas_reader: MetasDirReader,
        output_dir: str,
    ):
        """Runs embeddings extraction over meta_reader's data"""

        # Create output tree
        output_dir = Path(output_dir)
        ref_dir = output_dir / "rttm"
        emb_dir = output_dir / "embeddings"
        metas_dir = output_dir / "metas"
        ensure_dir(ref_dir)
        ensure_dir(emb_dir)

        # Run
        with get_rich_progress_bar("Embeddings extraction") as p:
            pbar = p.track(metas_reader, total=len(metas_reader))

            for item in pbar:
                file_name = get_pure_filename(item["wav"])
                print(f"Extraction from {file_name}")

                ref_annot = self._read_ref_rttm(item)

                # Extract
                embeddings, segments = self.extraction_step(
                    wav_path=item["wav"],
                    vad_path=item["vad"],
                    ref_annot=ref_annot,
                )

                # Save embeddings and rttm
                write_embeddings(embeddings, segments, emb_dir / f"{file_name}.npz")
                write_rttm(ref_annot, ref_dir / f"{file_name}.rttm")

        # Create metas
        create_metas_dir(metas_dir, [ref_dir, emb_dir])


In [17]:
import os
import yaml

import datetime
from pathlib import Path
from typing import Dict
from functools import cached_property

import hydra
from hydra.core.hydra_config import HydraConfig
import torch.nn as nn

from omegaconf import DictConfig, OmegaConf

from language_diarization.utils.meta import MetasDirReader
from language_diarization.utils.basic import ensure_dir
from language_diarization.utils.hydra import instantiate
from language_diarization.core.pipelines import *
from language_diarization.core.solvers import Solver
from language_diarization.utils.clearml_writer import ClearMlWriter


def parse_hydra_configs() -> Dict[str, str]:
    """
    Parse configs paths from runtime information.
    :return: dictionary with mapping config name into its local path
    """
    cfg_path = os.path.join(HydraConfig.get().runtime.config_sources[1].path,
                            HydraConfig.get().job.config_name + '.yaml')
    cfg_dict = {'main config': cfg_path}
    with open(cfg_path, "r") as stream:
        try:
            defaults = yaml.safe_load(stream)['defaults']
            for item in defaults:
                if type(item) == dict:
                    name = next(iter(item))
                    pth = item[name]
                    full_pth = os.path.join(HydraConfig.get().runtime.config_sources[1].path,
                                            name, pth + '.yaml')
                    cfg_dict[name] = full_pth
        except yaml.YAMLError as exc:
            print(exc)
    return cfg_dict


class Diarization:
    clearml_writer: ClearMlWriter = None
    clusterer: Solver = None
    pipeline: DiarizationPipeline = None
    clustering_module: nn.Module = None
    prediction_module: nn.Module = None
    meta_reader: MetasDirReader = None
    output_dir: str = None

    def __init__(self, config: DictConfig):
        self.config = config
        
        self.init_clusterer()
        self.init_clustering_module()
        
        if not self.are_extractors_equal:
            self.init_prediction_module()
            
        self.init_pipeline()
        self.init_meta_reader()
        self.create_report_dir()
        self.save_config()
        self.init_clearml()
        
    @cached_property
    def are_extractors_equal(self) -> bool:
        return self.config.prediction_module == self.config.clustering_module

    def init_clearml(self):
        if not self.config.clearml_off:
            self.clearml_writer = ClearMlWriter(self.config.project_name, self.config.task_name,
                                                continue_last_task=self.config.continue_last_task,
                                                base_name=self.config.base.name,
                                                output_dir=self.output_dir)
            try:
                cfg_dict = parse_hydra_configs()
                self.clearml_writer.connect_configs(cfg_dict)
            except RuntimeError:
                warnings.warn("Failed to log configs into CLearML.")

    def init_clustering_module(self):
        self.clustering_module = hydra.utils.instantiate(self.config.clustering_module)

    def init_prediction_module(self):
        self.prediction_module = hydra.utils.instantiate(self.config.prediction_module)

    def init_clusterer(self):
        self.clusterer = instantiate(self.config.clusterer)

    def init_pipeline(self): 
        if self.prediction_module is None:
            self.pipeline = DiarizationWithSameExtractor(
                clusterer=self.clusterer,
                clustering_module=self.clustering_module,
                **self.config.pipeline
            )
        else:
            self.pipeline = DiarizationWithDifferentExtractors(
                clusterer=self.clusterer,
                clustering_module=self.clustering_module,
                prediction_module=self.prediction_module,
                **self.config.pipeline
            )

    def init_meta_reader(self):
        try:
            self.meta_reader = MetasDirReader(metas_dir=self.config.base.metas_dir)
        except FileNotFoundError:
            print("One meta file not found in metas_dir. Presumably rttm.scp")
            print("Trying to load rttm_lang.scp instead.")
            self.meta_reader = MetasDirReader(
                metas_dir=self.config.base.metas_dir,
                required_metas=["wav.scp", "vad.scp", "rttm_lang.scp"],
            )

    def create_report_dir(self):
        base_name = self.config.base.name
        cluster_ext_name = self.config.clustering_module._target_.split(".")[-1].lower()
        predict_ext_name = self.config.prediction_module._target_.split(".")[-1].lower()
        solver_name = self.config.clusterer._target_.split(".")[-1].lower()
        clust_name = self.config.clusterer.clusterer_algo
        clust_name = clust_name._target_.split(".")[-1].lower()
        emb_type = self.config.pipeline.embeddings_type.lower()
        exp_name = f"{cluster_ext_name}-{emb_type}-{clust_name}+{solver_name}-{predict_ext_name}"
        curr_time = datetime.datetime.now().strftime("%d-%m-%Y_%H:%M:%S")

        self.output_dir = (
            Path(self.config.output_dir)
            / self.config.task_name
            / base_name
            / exp_name
            / curr_time
        )
        ensure_dir(self.output_dir)

    def save_config(self):
        with open(self.output_dir / "config.yaml", "w") as file:
            file.write(OmegaConf.to_yaml(self.config))

    def run(self):
        print(f"Processing {self.config.base.name}.\nOutput dir {self.output_dir}.")
        self.pipeline.run_over_metas(self.meta_reader, output_dir=self.output_dir, clearml_writer=self.clearml_writer)
        if self.clearml_writer:
            self.clearml_writer.close()


Языковая диаризация осуществляется через запуск пайплайна языковой диаризации.

In [18]:
!python3 -m language_diarization.cli.diarize

В результате получаем rttm-файл с разметкой.

По данной разметке можно получить моноязычные сегменты для транскрибации.

---